# `Instr._repr_html_` Test Notebook

This notebook builds an example instrument that exercises:
- `DECLARE` / `INITIALIZE` / `SAVE` / `FINALLY` C code blocks (collapsible via `<details>`)
- Instrument parameters
- Multiple component instances in the `TRACE` section (collapsible list)
- A `metadata` group
- An `InMemoryRegistry` (custom in-memory component definitions)
- A `RemoteRegistry` reference (displayed with clickable URL)

The last cell puts the `Instr` object on its own line so Jupyter calls `_repr_html_` automatically.

In [1]:
from mccode_antlr.assembler import Assembler
from mccode_antlr.reader.registry import InMemoryRegistry
from mccode_antlr import Flavor

In [2]:
# A minimal in-memory component so we don't need an external registry
simple_detector = """
DEFINE COMPONENT SimpleDetector
DEFINITION PARAMETERS ()
SETTING PARAMETERS (xwidth=0.1, yheight=0.1, string filename="det.dat")
OUTPUT PARAMETERS ()
SHARE
%{
%}
DECLARE
%{
  double counts;
%}
INITIALIZE
%{
  counts = 0;
%}
TRACE
%{
  SCATTER;
  counts += 1;
%}
SAVE
%{
  DETECTOR_OUT_2D("SimpleDetector", "X", "Y", -xwidth/2, xwidth/2,
                  -yheight/2, yheight/2, 100, 100, NULL, filename);
%}
FINALLY
%{
%}
END
"""

mem_registry = InMemoryRegistry('custom', SimpleDetector=simple_detector)

In [3]:
a = Assembler('example_instrument', flavor=Flavor.MCSTAS)

# Give the assembler access to our custom in-memory component
a.reader.registries.append(mem_registry)

# Instrument parameters
a.parameter('double E0 = 10')
a.parameter('double dE = 1')
a.parameter('double dist = 10')

# DECLARE block
a.declare('double v0;  /* neutron speed at source */')

# INITIALIZE block
a.initialize('v0 = SE2V * sqrt(E0);  /* SE2V converts meV to m/s */')

# SAVE block
a.save('fprintf(stdout, "Run complete. E0=%g meV\\n", E0);')

# Metadata group
a.metadata('instrument_info', 'text/plain',
           'Example instrument for testing mccode-antlr HTML output.\nBuilt programmatically with Assembler.')

# TRACE: component instances
origin = a.component('origin', 'Progress_bar', at=[0, 0, 0])

source = a.component(
    'source', 'Source_simple',
    at=([0, 0, 0], origin),
    parameters={
        'radius': 0.05,
        'yheight': 0,
        'xwidth': 0,
        'dist': 'dist',
        'focus_xw': 0.05,
        'focus_yh': 0.05,
        'E0': 'E0',
        'dE': 'dE',
    },
)

guide = a.component(
    'guide', 'Guide_gravity',
    at=([0, 0, 2], source),
    parameters={
        'w1': 0.05,
        'h1': 0.05,
        'w2': 0.03,
        'h2': 0.03,
        'l': 10,
        'm': 3.5,
    },
)

slit = a.component(
    'slit', 'Slit',
    at=([0, 0, 0.1], guide),
    parameters={'xwidth': 0.02, 'yheight': 0.02},
)

sample = a.component(
    'sample', 'V_sample',
    at=([0, 0, 1], slit),
    parameters={
        'radius': 0.005,
        'yheight': 0.05,
        'focus_r': 0.3,
        'target_index': 1,
    },
)

# Use our custom in-memory component
detector = a.component(
    'detector', 'SimpleDetector',
    at=([0, 0, 0.5], sample),
    parameters={'xwidth': 0.3, 'yheight': 0.3, 'filename': '"psd.dat"'},
)

instr = a.instrument

2026-03-05 15:11:11.844 | DEBUG    | mccode_antlr.reader.reader:add_component:243 - Component cache hit: /home/g/.cache/mccodeantlr/mcstas/v3.5.31/mcstas-comps/misc/Progress_bar.comp
2026-03-05 15:11:11.858 | DEBUG    | mccode_antlr.reader.reader:add_component:243 - Component cache hit: /home/g/.cache/mccodeantlr/mcstas/v3.5.31/mcstas-comps/sources/Source_simple.comp
2026-03-05 15:11:11.864 | DEBUG    | mccode_antlr.reader.reader:add_component:243 - Component cache hit: /home/g/.cache/mccodeantlr/mcstas/v3.5.31/mcstas-comps/optics/Guide_gravity.comp
2026-03-05 15:11:11.869 | DEBUG    | mccode_antlr.reader.reader:add_component:243 - Component cache hit: /home/g/.cache/mccodeantlr/mcstas/v3.5.31/mcstas-comps/optics/Slit.comp
2026-03-05 15:11:11.874 | DEBUG    | mccode_antlr.reader.reader:add_component:243 - Component cache hit: /home/g/.cache/mccodeantlr/mcstas/v3.5.31/mcstas-comps/obsolete/V_sample.comp


## Display the instrument

Placing `instr` on the last line of a code cell triggers Jupyter's rich display — it calls `_repr_html_()` automatically.

Expect to see:
- `DECLARE`, `INITIALIZE`, `SAVE` code blocks collapsed under `<details>` (click to expand)
- `TRACE` section as a collapsible open list of components
- Registry headers with clickable links (for remote registries)
- `InMemoryRegistry: custom (1 components)` label for the in-memory registry

In [4]:
instr

Instr(name='example_instrument', source='interactive', parameters=(InstrumentParameter(name='E0', unit=None, value=scalar DataType.float 10), InstrumentParameter(name='dE', unit=None, value=scalar DataType.float 1), InstrumentParameter(name='dist', unit=None, value=scalar DataType.float 10)), metadata=(MetaData(source=DataSource(_type=<Type.Instrument: 3>, _name=None), name='instrument_info', mimetype='text/plain', value='Example instrument for testing mccode-antlr HTML output.\nBuilt programmatically with Assembler.'),), components=(Instance(origin, Progress_bar), Instance(source, Source_simple), Instance(guide, Guide_gravity), Instance(slit, Slit), Instance(sample, V_sample), Instance(detector, SimpleDetector)), included=(), user=(), declare=(RawC(filename=None, line=-1, source='double v0;  /* neutron speed at source */', translated=None),), initialize=(RawC(filename=None, line=-1, source='v0 = SE2V * sqrt(E0);  /* SE2V converts meV to m/s */', translated=None),), save=(RawC(filename=None, line=-1, source='fprintf(stdout, "Run complete. E0=%g meV\\n", E0);', translated=None),), final=(), registries=[<mccode_antlr.reader.registry.GitHubRegistry object at 0x7f4990dad2b0>, <mccode_antlr.reader.registry.GitHubRegistry object at 0x7f4990f07ed0>, <mccode_antlr.reader.registry.InMemoryRegistry object at 0x7f49912670e0>], dependency=(), flow_edges=(FlowEdgeRecord(src='origin', dst='source', edge=SequentialEdge(when=None)), FlowEdgeRecord(src='source', dst='guide', edge=SequentialEdge(when=None)), FlowEdgeRecord(src='guide', dst='slit', edge=SequentialEdge(when=None)), FlowEdgeRecord(src='slit', dst='sample', edge=SequentialEdge(when=None)), FlowEdgeRecord(src='sample', dst='detector', edge=SequentialEdge(when=None))))